In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [2]:
# Dataset
text = """
Artificial intelligence is transforming the world by enabling machines to learn from data and make decisions.
Machine learning is a subset of artificial intelligence that focuses on building systems that improve automatically through experience.
Deep learning is a specialized branch of machine learning that uses neural networks with multiple layers.
Natural language processing allows machines to understand and generate human language.
Applications of artificial intelligence include chatbots, recommendation systems, image recognition, and autonomous vehicles.
"""

# Convert to lowercase
text = text.lower()

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

In [3]:
input_sequences = []

for line in text.split("."):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Padding sequences
max_sequence_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

# Split predictors and label
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# Convert labels to categorical
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

In [4]:
model = Sequential()
model.add(Embedding(total_words, 64, input_length=max_sequence_len - 1))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
history = model.fit(
    X,
    y,
    epochs=200,
    verbose=1
)

Epoch 1/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.0112 - loss: 4.0075  
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.1015 - loss: 3.9970
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0904 - loss: 3.9859
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.0636 - loss: 3.9754
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.0680 - loss: 3.9522
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.0413 - loss: 3.9150
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0413 - loss: 3.8470
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0491 - loss: 3.8002
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0569 - loss: 3.7992
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0747 - loss: 3.7715
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0786 - loss: 3.7361
Epoch 12/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.1060 - 

In [6]:
def predict_next_word(model, tokenizer, text, max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text.lower()])[0]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = np.argmax(model.predict(token_list), axis=-1)

    for word, index in tokenizer.word_index.items():
        if index == predicted:
            return word

# Test examples
seed_text = "artificial intelligence"
next_word = predict_next_word(model, tokenizer, seed_text, max_sequence_len)

print(f"Input Text: {seed_text}")
print(f"Predicted Next Word: {next_word}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step
Input Text: artificial intelligence
Predicted Next Word: is


In [9]:
import os

os.listdir()


['.config', 'next_word_prediction_lstm.h5', 'sample_data']

In [10]:
model.save("next_word_prediction_lstm.keras")

In [11]:
os.listdir()

['.config',
 'next_word_prediction_lstm.keras',
 'next_word_prediction_lstm.h5',
 'sample_data']

In [12]:
from tensorflow.keras.models import load_model

model = load_model("next_word_prediction_lstm.keras")
print("Model loaded successfully.")

Model loaded successfully.


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [14]:
seed_text = "machine learning is"
next_word = predict_next_word(model, tokenizer, seed_text, max_sequence_len)

print("Input:", seed_text)
print("Predicted Next Word:", next_word)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step
Input: machine learning is
Predicted Next Word: a
